<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/02_PDF_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: PDF Preprocessing and Dataset Stratification
Based on Phase 1 EDA, we know:
- There are no corrupted PDFs and no exact duplicates.
- Classes are highly imbalanced (e.g., BPO has only 22 instances).

**Goals:**
- Keep original PDFs immutable.
- Build a structured metadata catalog (mapping files to categories).
- Perform a strict stratified Train/Val/Test split to avoid data leakage and ensure minority classes are represented.

In [ ]:
!pip install huggingface_hub pandas scikit-learn tqdm datasets

In [ ]:
import os
import pandas as pd
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm

# Download the raw dataset
print("Downloading dataset files from Hugging Face...")
dataset_path = snapshot_download(repo_id="BassemRamdan/data", repo_type="dataset")
print(f"Dataset downloaded to: {dataset_path}")

In [ ]:
print("Building Metadata Catalog...")
data = []
categories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.git')]

for category in tqdm(categories, desc="Scanning Categories"):
    cat_path = os.path.join(dataset_path, category)
    for filename in os.listdir(cat_path):
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(cat_path, filename)
            # We record relative paths to maintain immutability and portability
            relative_path = os.path.join(category, filename)
            data.append({
                "relative_path": relative_path,
                "category": category,
                "filename": filename
            })

metadata_df = pd.DataFrame(data)
print(f"\nMetadata Shape: {metadata_df.shape}")
metadata_df.head()

In [ ]:
print("Performing Stratified Split (70% Train, 15% Val, 15% Test)...")
# First split: 70% Train, 30% Temp (Val+Test)
train_df, temp_df = train_test_split(metadata_df, test_size=0.3, random_state=42, stratify=metadata_df['category'])

# Second split: 50% of Temp into Val, 50% into Test (i.e. 15% / 15% of total)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['category'])

print(f"Train set: {len(train_df)} resumes")
print(f"Validation set: {len(val_df)} resumes")
print(f"Test set: {len(test_df)} resumes")

# Add split labels to the dataframe
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

final_metadata_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)

print("\nDistribution Check for Smallest Class (BPO):")
print(final_metadata_df[final_metadata_df['category'] == 'BPO']['split'].value_counts())

In [ ]:
# Export metadata locally
final_metadata_df.to_csv("resume_metadata_split.csv", index=False)
print("Metadata saved to resume_metadata_split.csv")

# Optionally push to a new Hugging Face dataset to standardize the pipeline
hf_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df.drop(columns=['split']).reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.drop(columns=['split']).reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.drop(columns=['split']).reset_index(drop=True))
})

print(hf_dataset)
# To push to Hub, uncomment the following:
# hf_dataset.push_to_hub("BassemRamdan/resume-metadata-splits")